<table style="width:100%; border-collapse:collapse; border-spacing:0; border:none; border-bottom:2px solid #e0e0e0; margin-bottom:16px; padding:0; margin-top:0;">
  <tr style="border:none; padding:0; margin:0;">
    <td style="width:144px; padding:16px 24px 16px 0; vertical-align:middle; border:none; outline:none; box-shadow:none;">
      <img src="ams_group_tui.png" width="120" style="border-radius:12px; display:block; margin:0;"/>
    </td>
    <td style="padding:16px 0 16px 0; vertical-align:middle; border:none; outline:none; box-shadow:none;">
      <h1 style="margin:0 0 4px 0; padding:0;">Multimediaprogrammierung</h1>
      <h2 style="margin:0 0 8px 0; padding:0;">Chapter 01: Web Audio API</h2>
      <p style="margin:0; padding:0;">
        <b>Institut für Medientechnik – Technische Universität Ilmenau</b><br>
        <a href="https://www.tu-ilmenau.de/universitaet/fakultaeten/fakultaet-elektrotechnik-und-informationstechnik/profil/institute-und-fachgebiete/fachgebiet-angewandte-mediensysteme"
           target="_blank">Institut für Medientechnik – AMS Group</a>
      </p>
      <p style="margin:4px 0 0 0; padding:0;">
        <b>Fachgebietsleiter:</b> Prof. Dr.-Ing. Gerald Schuller<br>
        <b>Wissenschaftlicher Mitarbeiter:</b> Renato Fera P. <a href="https://www.linkedin.com/in/renato-profeta/" target="_blank">LinkedIn</a>
      </p>
    </td>
  </tr>
</table>


# Chapter 01: Web Audio API

The **Web Audio API** is a high-level JavaScript API built into every modern browser that gives you full control over audio processing and synthesis — without any plugins or external libraries. It was designed for games, interactive applications and creative tools, but it is also an excellent environment for learning the fundamentals of digital audio: oscillators, filters, gain, panning, analysis and effects.

What makes it special is its **audio graph model**: instead of writing raw DSP code, you connect nodes together into a signal chain. Each node does one job — generate a tone, apply a filter, control the volume — and you connect them like modules in a synthesizer. The browser handles the real-time audio engine underneath.

This chapter covers the core concepts of the Web Audio API from the ground up, with interactive demos that run directly in the browser.

### What this chapter covers

| #  | Topic | What you will learn |
| -- | ----- | ------------------- |
| 1  | The Audio Graph | How the Web Audio API models signal flow |
| 2  | AudioContext | The entry point for all audio operations |
| 3  | OscillatorNode | Generating tones and waveforms |
| 4  | GainNode | Controlling volume and amplitude envelopes |
| 5  | BiquadFilterNode | Filtering: low-pass, high-pass, band-pass |
| 6  | StereoPannerNode | Positioning sound in the stereo field |
| 7  | AnalyserNode | Visualizing waveforms and frequency spectra |
| 8  | Loading and Playing Audio Files | Fetch, decode and play back audio from files |
| 9  | Interactive Demos | Oscillator, visualizer and synthesizer demos |
| 10 | Further Resources | MDN reference, specifications and tutorials |


---

## 1. The Audio Graph

The central idea of the Web Audio API is the **audio graph**: a directed network of processing nodes through which audio flows from sources to a destination. You build this graph by creating nodes and connecting their outputs to the inputs of other nodes. The browser then runs the entire graph in a real-time audio thread, sample by sample.

Every audio graph has three kinds of nodes:

| Node type | Role | Examples |
| --------- | ---- | -------- |
| **Source nodes** | Generate or supply audio signal | `OscillatorNode`, `AudioBufferSourceNode`, `MediaElementAudioSourceNode` |
| **Processing nodes** | Transform the signal | `GainNode`, `BiquadFilterNode`, `ConvolverNode`, `StereoPannerNode`, `AnalyserNode` |
| **Destination node** | The final output (speakers) | `AudioContext.destination` |

A simple signal chain looks like this:

```
OscillatorNode  →  GainNode  →  AudioContext.destination
```

Nodes connect with the `.connect()` method:

```javascript
oscillator.connect(gainNode);
gainNode.connect(audioCtx.destination);
```

The graph can branch and merge freely. You can connect one source to multiple processing chains, or feed multiple sources into the same destination. The `AnalyserNode` is a useful example of branching: it passes audio through unchanged but also exposes the signal for visualization, so you can tap into any point in the graph without affecting the sound.

```
OscillatorNode  →  GainNode  →  AnalyserNode  →  AudioContext.destination
                                     ↓
                              (canvas oscilloscope)
```

<p style="margin:12px 0 0 0;">
  <a href="https://developer.mozilla.org/en-US/docs/Web/API/Web_Audio_API/Basic_concepts_behind_Web_Audio_API" target="_blank" style="display:inline-block; padding:6px 14px; margin:4px 8px 4px 0; border:1px solid #1a73e8; border-radius:20px; text-decoration:none; font-weight:600; font-size:0.95em;">Audio Graph – MDN &#8599;</a>
  <a href="https://developer.mozilla.org/en-US/docs/Web/API/AudioNode" target="_blank" style="display:inline-block; padding:6px 14px; margin:4px 8px 4px 0; border:1px solid #1a73e8; border-radius:20px; text-decoration:none; font-weight:600; font-size:0.95em;">AudioNode – MDN &#8599;</a>
</p>


---

## 2. AudioContext

The `AudioContext` is the root object of the Web Audio API. It represents the audio hardware of the device and controls the **sample rate**, the **current time** and the **destination** (your speakers or headphones). Every node you create belongs to exactly one context, and all nodes in a graph must share the same context.

You create an `AudioContext` with a single line:

```javascript
const audioCtx = new AudioContext();
```

**Important — the autoplay policy.** Modern browsers block audio from starting without a user gesture (a click, tap or keypress). If you try to create or resume an `AudioContext` before the user has interacted with the page, the browser will either refuse or suspend it. Always start audio in response to a button click.

```javascript
button.addEventListener('click', () => {
    const audioCtx = new AudioContext();  // safe: inside a user gesture
    // ... create nodes, connect, start
});
```

A few key properties of `AudioContext`:

| Property / Method | What it does |
| ----------------- | ------------ |
| `audioCtx.sampleRate` | The sample rate in Hz (typically 44100 or 48000) |
| `audioCtx.currentTime` | The running playback clock in seconds (read-only, high-precision) |
| `audioCtx.destination` | The final output node (connects to speakers) |
| `audioCtx.state` | `"suspended"`, `"running"` or `"closed"` |
| `audioCtx.resume()` | Resume a suspended context |
| `audioCtx.close()` | Release all resources |

The `currentTime` clock is very important for scheduling: you can tell a node to start or change a parameter at a precise future time, enabling glitch-free audio transitions and rhythm-accurate sequencing.

<p style="margin:12px 0 0 0;">
  <a href="https://developer.mozilla.org/en-US/docs/Web/API/AudioContext" target="_blank" style="display:inline-block; padding:6px 14px; margin:4px 8px 4px 0; border:1px solid #1a73e8; border-radius:20px; text-decoration:none; font-weight:600; font-size:0.95em;">AudioContext – MDN &#8599;</a>
  <a href="https://developer.mozilla.org/en-US/docs/Web/API/AudioContext/currentTime" target="_blank" style="display:inline-block; padding:6px 14px; margin:4px 8px 4px 0; border:1px solid #1a73e8; border-radius:20px; text-decoration:none; font-weight:600; font-size:0.95em;">currentTime – MDN &#8599;</a>
</p>


---

## 3. OscillatorNode

An `OscillatorNode` is a **source node** that generates a periodic waveform at a given frequency. It is the most fundamental building block for sound synthesis in the browser — no audio file needed.

You create and start an oscillator like this:

```javascript
const oscillator = audioCtx.createOscillator();
oscillator.type = 'sine';        // waveform shape
oscillator.frequency.value = 440; // frequency in Hz (A4)
oscillator.connect(audioCtx.destination);
oscillator.start();              // begin generating audio
oscillator.stop(audioCtx.currentTime + 2); // stop after 2 seconds
```

### Waveform types

The `type` property selects the waveform. Each has a different harmonic content and timbre:

| Type | Harmonics | Sound character |
| ---- | --------- | --------------- |
| `sine` | Fundamental only | Pure, smooth, flute-like |
| `square` | Odd harmonics (strong) | Hollow, buzzy, clarinet-like |
| `sawtooth` | All harmonics | Bright, rich, string/brass-like |
| `triangle` | Odd harmonics (weak) | Soft, mellow, somewhere between sine and square |
| `custom` | User-defined via `PeriodicWave` | Any waveform you can describe with Fourier coefficients |

### AudioParam and smooth transitions

Properties like `frequency` and `detune` are **AudioParam** objects, not plain numbers. This lets you schedule precise changes in time, which is essential for smooth, glitch-free audio:

```javascript
// Ramp the frequency smoothly from 440 Hz to 880 Hz over 1 second
oscillator.frequency.linearRampToValueAtTime(880, audioCtx.currentTime + 1);

// Set a value immediately but with a short smoothing time constant
oscillator.frequency.setTargetAtTime(660, audioCtx.currentTime, 0.01);
```

<p style="margin:12px 0 0 0;">
  <a href="https://developer.mozilla.org/en-US/docs/Web/API/OscillatorNode" target="_blank" style="display:inline-block; padding:6px 14px; margin:4px 8px 4px 0; border:1px solid #1a73e8; border-radius:20px; text-decoration:none; font-weight:600; font-size:0.95em;">OscillatorNode – MDN &#8599;</a>
  <a href="https://developer.mozilla.org/en-US/docs/Web/API/AudioParam" target="_blank" style="display:inline-block; padding:6px 14px; margin:4px 8px 4px 0; border:1px solid #1a73e8; border-radius:20px; text-decoration:none; font-weight:600; font-size:0.95em;">AudioParam – MDN &#8599;</a>
</p>


---

## 4. GainNode

A `GainNode` multiplies the amplitude of its input signal by a factor called **gain**. A gain of `1.0` means no change, `0.5` halves the amplitude (roughly −6 dB), and `0.0` produces silence. Values above `1.0` amplify the signal.

```javascript
const gainNode = audioCtx.createGain();
gainNode.gain.value = 0.5;  // 50% amplitude

oscillator.connect(gainNode);
gainNode.connect(audioCtx.destination);
```

### Amplitude envelopes (ADSR)

`GainNode` is also how you shape the **amplitude envelope** of a sound — the way the volume rises and falls over time. In synthesis this is described with the **ADSR** model:

| Stage | Meaning | What it sounds like |
| ----- | ------- | ------------------- |
| **A**ttack | Time to reach peak amplitude | How quickly the note "hits" |
| **D**ecay | Time to fall from peak to sustain level | The initial drop after the hit |
| **S**ustain | The level held while the key is pressed | The steady-state volume |
| **R**elease | Time to fade to silence after the key is released | How long the tail rings out |

You implement an ADSR envelope using `AudioParam` scheduling on the gain:

```javascript
const now = audioCtx.currentTime;
const gain = gainNode.gain;

gain.setValueAtTime(0, now);                        // start at silence
gain.linearRampToValueAtTime(1.0, now + 0.01);      // Attack: 10 ms
gain.linearRampToValueAtTime(0.7, now + 0.1);       // Decay: 90 ms → sustain level
// ... key held ...
gain.setValueAtTime(0.7, now + 0.5);                // begin Release
gain.linearRampToValueAtTime(0.0, now + 0.7);       // Release: 200 ms to silence
```

<p style="margin:12px 0 0 0;">
  <a href="https://developer.mozilla.org/en-US/docs/Web/API/GainNode" target="_blank" style="display:inline-block; padding:6px 14px; margin:4px 8px 4px 0; border:1px solid #1a73e8; border-radius:20px; text-decoration:none; font-weight:600; font-size:0.95em;">GainNode – MDN &#8599;</a>
  <a href="https://developer.mozilla.org/en-US/docs/Web/API/AudioParam#methods" target="_blank" style="display:inline-block; padding:6px 14px; margin:4px 8px 4px 0; border:1px solid #1a73e8; border-radius:20px; text-decoration:none; font-weight:600; font-size:0.95em;">AudioParam scheduling – MDN &#8599;</a>
</p>


---

## 5. BiquadFilterNode

A `BiquadFilterNode` applies a **second-order IIR filter** to the audio signal. Filters shape the frequency content of a sound by boosting or cutting certain frequency ranges. They are used in equalizers, synthesizer filter sweeps, speaker crossovers and countless other applications.

```javascript
const filter = audioCtx.createBiquadFilter();
filter.type = 'lowpass';
filter.frequency.value = 1000;  // cutoff frequency in Hz
filter.Q.value = 1;             // resonance (Q factor)

oscillator.connect(filter);
filter.connect(audioCtx.destination);
```

### Filter types

| Type | What it does |
| ---- | ------------ |
| `lowpass` | Passes frequencies below the cutoff, attenuates above |
| `highpass` | Passes frequencies above the cutoff, attenuates below |
| `bandpass` | Passes a band of frequencies around the cutoff |
| `notch` | Attenuates a narrow band (the opposite of bandpass) |
| `peaking` | Boosts or cuts around the center frequency (like an equalizer band) |
| `lowshelf` | Boosts or cuts all frequencies below the shelf frequency |
| `highshelf` | Boosts or cuts all frequencies above the shelf frequency |

### The Q factor

The `Q` (quality factor) controls the **resonance** of the filter — how sharply it acts around the cutoff frequency. A high Q on a low-pass filter produces a characteristic resonant peak just before the cutoff, which is the classic "synth filter sweep" sound.

<p style="margin:12px 0 0 0;">
  <a href="https://developer.mozilla.org/en-US/docs/Web/API/BiquadFilterNode" target="_blank" style="display:inline-block; padding:6px 14px; margin:4px 8px 4px 0; border:1px solid #1a73e8; border-radius:20px; text-decoration:none; font-weight:600; font-size:0.95em;">BiquadFilterNode – MDN &#8599;</a>
  <a href="https://developer.mozilla.org/en-US/docs/Web/API/BiquadFilterNode/type" target="_blank" style="display:inline-block; padding:6px 14px; margin:4px 8px 4px 0; border:1px solid #1a73e8; border-radius:20px; text-decoration:none; font-weight:600; font-size:0.95em;">Filter types – MDN &#8599;</a>
</p>


---

## 6. StereoPannerNode

A `StereoPannerNode` positions a mono or stereo signal in the **stereo field** using a single `pan` parameter. A value of `-1` sends the signal fully to the left channel, `0` centers it, and `+1` sends it fully to the right.

```javascript
const panner = audioCtx.createStereoPanner();
panner.pan.value = -0.5;  // slightly left of center

oscillator.connect(gainNode);
gainNode.connect(panner);
panner.connect(audioCtx.destination);
```

Like all AudioParam values, `pan` can be automated over time. You can use `linearRampToValueAtTime` or `setTargetAtTime` to animate the pan position, creating a moving or sweeping stereo effect.

For more advanced 3D spatial audio positioning — with distance, room modelling and head-related transfer functions (HRTF) — the Web Audio API also provides a `PannerNode`. The `StereoPannerNode` is the simpler and more common choice for stereo mixing.

<p style="margin:12px 0 0 0;">
  <a href="https://developer.mozilla.org/en-US/docs/Web/API/StereoPannerNode" target="_blank" style="display:inline-block; padding:6px 14px; margin:4px 8px 4px 0; border:1px solid #1a73e8; border-radius:20px; text-decoration:none; font-weight:600; font-size:0.95em;">StereoPannerNode – MDN &#8599;</a>
  <a href="https://developer.mozilla.org/en-US/docs/Web/API/PannerNode" target="_blank" style="display:inline-block; padding:6px 14px; margin:4px 8px 4px 0; border:1px solid #1a73e8; border-radius:20px; text-decoration:none; font-weight:600; font-size:0.95em;">PannerNode (3D) – MDN &#8599;</a>
</p>


---

## 7. AnalyserNode

The `AnalyserNode` is a **pass-through processing node** that exposes the audio signal for real-time analysis without modifying it. You insert it into the signal chain and then read data from it in a `requestAnimationFrame` loop to drive a visualization.

```javascript
const analyser = audioCtx.createAnalyser();
analyser.fftSize = 2048;  // size of the FFT; determines frequency resolution

oscillator.connect(gainNode);
gainNode.connect(analyser);
analyser.connect(audioCtx.destination);
```

Two types of data are available:

| Method | Data | Use case |
| ------ | ---- | -------- |
| `getFloatTimeDomainData(array)` | Waveform samples (−1 to +1) | Oscilloscope |
| `getByteFrequencyData(array)` | Frequency magnitudes (0–255) | Spectrum analyser / equalizer bar display |

### Drawing a waveform on a canvas

```javascript
const bufferLength = analyser.frequencyBinCount; // fftSize / 2
const dataArray = new Float32Array(bufferLength);

function draw() {
    requestAnimationFrame(draw);
    analyser.getFloatTimeDomainData(dataArray);

    ctx.clearRect(0, 0, canvas.width, canvas.height);
    ctx.beginPath();
    const sliceWidth = canvas.width / bufferLength;
    let x = 0;
    for (let i = 0; i < bufferLength; i++) {
        const y = (dataArray[i] * 0.9 + 1) / 2 * canvas.height;
        i === 0 ? ctx.moveTo(x, y) : ctx.lineTo(x, y);
        x += sliceWidth;
    }
    ctx.stroke();
}
draw();
```

The `fftSize` must be a power of two between 32 and 32768. A larger `fftSize` gives better **frequency resolution** in the spectrum display but uses more memory and CPU.

<p style="margin:12px 0 0 0;">
  <a href="https://developer.mozilla.org/en-US/docs/Web/API/AnalyserNode" target="_blank" style="display:inline-block; padding:6px 14px; margin:4px 8px 4px 0; border:1px solid #1a73e8; border-radius:20px; text-decoration:none; font-weight:600; font-size:0.95em;">AnalyserNode – MDN &#8599;</a>
  <a href="https://developer.mozilla.org/en-US/docs/Web/API/AnalyserNode/fftSize" target="_blank" style="display:inline-block; padding:6px 14px; margin:4px 8px 4px 0; border:1px solid #1a73e8; border-radius:20px; text-decoration:none; font-weight:600; font-size:0.95em;">fftSize – MDN &#8599;</a>
</p>


---

## 8. Loading and Playing Audio Files

To play an audio file with the Web Audio API you need to load it, decode it into raw PCM samples, and then play it through an `AudioBufferSourceNode`. This gives you full control over playback speed, looping, scheduling and further processing through the audio graph.

### Step 1 — Fetch and decode

```javascript
async function loadAudio(url) {
    const response = await fetch(url);
    const arrayBuffer = await response.arrayBuffer();
    const audioBuffer = await audioCtx.decodeAudioData(arrayBuffer);
    return audioBuffer;  // decoded PCM samples in an AudioBuffer
}
```

`decodeAudioData` accepts compressed formats like MP3, OGG and WAV, and returns an `AudioBuffer` object holding the raw floating-point samples at the context's sample rate.

### Step 2 — Play with AudioBufferSourceNode

```javascript
const buffer = await loadAudio('sound.mp3');

const source = audioCtx.createBufferSource();
source.buffer = buffer;
source.loop = false;
source.connect(audioCtx.destination);
source.start();  // play immediately
```

An `AudioBufferSourceNode` is **single-use**: once it has played, it is finished. To play the same sound again you create a new source node and assign the same buffer to it. The buffer itself is reusable and is decoded only once.

### AudioBuffer properties

| Property | What it gives you |
| -------- | ----------------- |
| `buffer.duration` | Length in seconds |
| `buffer.sampleRate` | Sample rate in Hz |
| `buffer.numberOfChannels` | 1 = mono, 2 = stereo |
| `buffer.getChannelData(n)` | `Float32Array` of PCM samples for channel n |

<p style="margin:12px 0 0 0;">
  <a href="https://developer.mozilla.org/en-US/docs/Web/API/AudioBufferSourceNode" target="_blank" style="display:inline-block; padding:6px 14px; margin:4px 8px 4px 0; border:1px solid #1a73e8; border-radius:20px; text-decoration:none; font-weight:600; font-size:0.95em;">AudioBufferSourceNode – MDN &#8599;</a>
  <a href="https://developer.mozilla.org/en-US/docs/Web/API/BaseAudioContext/decodeAudioData" target="_blank" style="display:inline-block; padding:6px 14px; margin:4px 8px 4px 0; border:1px solid #1a73e8; border-radius:20px; text-decoration:none; font-weight:600; font-size:0.95em;">decodeAudioData – MDN &#8599;</a>
  <a href="https://developer.mozilla.org/en-US/docs/Web/API/AudioBuffer" target="_blank" style="display:inline-block; padding:6px 14px; margin:4px 8px 4px 0; border:1px solid #1a73e8; border-radius:20px; text-decoration:none; font-weight:600; font-size:0.95em;">AudioBuffer – MDN &#8599;</a>
</p>


---

## 9. Interactive Demos

The following demos let you hear and explore the concepts covered in this chapter. Because the Web Audio API requires a full browser environment with user-gesture controls and a canvas for visualization, the demos are provided as standalone HTML files.

**How to open them:**
- **Locally:** The HTML files are in the same folder as this notebook. Open them directly in your browser.
- **On GitHub:** Use the links below, which render the files via [htmlpreview.github.io](https://htmlpreview.github.io).

### Demo 1 — Oscillator

Explore the four standard waveform types (sine, square, sawtooth, triangle) and adjust frequency and gain in real time. A live oscilloscope shows the waveform as audio is generated.

<p style="margin:12px 0 0 0;">
  <a href="https://htmlpreview.github.io/?url=https://raw.githubusercontent.com/TUIlmenauAMS/multimedia_programming_seminars/main/01_webaudio_oscillator.html" target="_blank" style="display:inline-block; padding:6px 14px; margin:4px 8px 4px 0; border:1px solid #1a73e8; border-radius:20px; text-decoration:none; font-weight:600; font-size:0.95em;">Open Oscillator Demo &#8599;</a>
</p>

---

### Demo 2 — Visualizer

See the `AnalyserNode` in action. This demo shows both a real-time waveform (oscilloscope) and a frequency spectrum display (FFT), side by side, for an oscillator signal.

<p style="margin:12px 0 0 0;">
  <a href="https://htmlpreview.github.io/?url=https://raw.githubusercontent.com/TUIlmenauAMS/multimedia_programming_seminars/main/01_webaudio_visualizer.html" target="_blank" style="display:inline-block; padding:6px 14px; margin:4px 8px 4px 0; border:1px solid #1a73e8; border-radius:20px; text-decoration:none; font-weight:600; font-size:0.95em;">Open Visualizer Demo &#8599;</a>
</p>

---

### Demo 3 — Synthesizer

A simple polyphonic synthesizer that combines an oscillator, a `BiquadFilterNode` and a `GainNode` with an ADSR envelope. Play notes with the on-screen keyboard and experiment with the filter cutoff, resonance and envelope controls.

<p style="margin:12px 0 0 0;">
  <a href="https://htmlpreview.github.io/?url=https://raw.githubusercontent.com/TUIlmenauAMS/multimedia_programming_seminars/main/01_webaudio_synth.html" target="_blank" style="display:inline-block; padding:6px 14px; margin:4px 8px 4px 0; border:1px solid #1a73e8; border-radius:20px; text-decoration:none; font-weight:600; font-size:0.95em;">Open Synthesizer Demo &#8599;</a>
</p>


---

## 10. Further Resources

The Web Audio API is a large specification with many more nodes and features beyond what this chapter covers. The resources below are good next steps.

| Resource | What it covers |
| -------- | -------------- |
| **MDN Web Audio API guide** | The most complete and practical reference, with examples for every node |
| **W3C Web Audio API specification** | The formal specification; useful for precise behaviour and edge cases |
| **Web Audio API book (O'Reilly)** | A free online book covering synthesis, effects and game audio in depth |
| **Web Audio Weekly** | A newsletter and archive of demos, libraries and articles from the community |

The video tutorial below, _Web Audio API Tutorial_, is the recommended companion video for this chapter. It walks through the core concepts hands-on, from creating an `AudioContext` to building a small synthesizer.

<p style="margin:12px 0 0 0;">
  <a href="https://developer.mozilla.org/en-US/docs/Web/API/Web_Audio_API" target="_blank" style="display:inline-block; padding:6px 14px; margin:4px 8px 4px 0; border:1px solid #1a73e8; border-radius:20px; text-decoration:none; font-weight:600; font-size:0.95em;">MDN Web Audio API &#8599;</a>
  <a href="https://www.w3.org/TR/webaudio/" target="_blank" style="display:inline-block; padding:6px 14px; margin:4px 8px 4px 0; border:1px solid #1a73e8; border-radius:20px; text-decoration:none; font-weight:600; font-size:0.95em;">W3C Specification &#8599;</a>
  <a href="https://webaudioapi.com/book/" target="_blank" style="display:inline-block; padding:6px 14px; margin:4px 8px 4px 0; border:1px solid #1a73e8; border-radius:20px; text-decoration:none; font-weight:600; font-size:0.95em;">Web Audio API Book &#8599;</a>
</p>

<p align="center" style="margin:20px 0;">
  <a href="https://www.youtube.com/watch?v=FCiQg7G-HF8" target="_blank">
    <img src="https://img.youtube.com/vi/FCiQg7G-HF8/maxresdefault.jpg" width="640" alt="Web Audio API Tutorial"
         style="max-width:100%; border-radius:14px; box-shadow:0 4px 18px rgba(0,0,0,0.25);"/>
  </a>
</p>
<p align="center" style="margin:0;"><i>Click the image to watch the tutorial on YouTube.</i></p>


---

<div style="text-align:center; padding:18px 0 8px 0; margin-top:24px; border-top:2px solid #e0e0e0;">
  <p style="margin:0 0 6px 0;">
    <b>Multimediaprogrammierung</b> &middot; Chapter 01: Web Audio API<br>
    Institut für Medientechnik &ndash; Technische Universität Ilmenau
  </p>
  <p style="margin:0; font-size:0.95em;">
    Notebook created by <b>Renato Fera P.</b> &middot;
    <a href="https://www.linkedin.com/in/renato-profeta/" target="_blank">LinkedIn</a>
  </p>
</div>
